# Citation Network Analysis with Author Influence
Fariha Adil, 2375026

## About
The goal of this analysis is to build a citation network to identify influential papers and authors. Graph-based metrics like PageRank and in-degree centrality are used to measure influence, which is then tracked over time to reveal how impact evolves and which papers represent true breakthroughs in computer science research.

## Tasks
- Build a directed citation network where nodes are papers and edges represent citation relationships
- Use graph metrics (in-degree centrality, PageRank, betweenness centrality) to identify influential papers and authors
- Combine influence scores with temporal data to track how author and paper impact evolves over time
- Identify breakthrough papers that shifted research directions using citation velocity and PageRank

## Motivations
- Understand which papers and authors have truly shaped computer science research beyond simple citation counts
- Identify emerging researchers and paradigm-shifting work that may go unnoticed by raw citation metrics alone
- Track how influence evolves: does an author's impact grow over time or fade? Do seminal papers maintain relevance?
- Discover research momentum: which areas are gaining traction vs. declining

## Challenges
- Computational complexity: building and analyzing networks with millions of nodes and edges requires efficient graph algorithms and working with filtered subsets
- Incomplete citation data: not all papers cite all prior relevant work; citations reflect only the subset recorded in DBLP
- Temporal dynamics: influence changes over time; time windows must be carefully defined and papers with delayed impact must be accounted for
- Bias in metrics: PageRank and centrality favor well-established papers; must balance with novelty metrics like citation velocity
- Author disambiguation: the same name can refer to different authors, making author-level influence aggregation imprecise

In [7]:
"""Import Dependencies"""
import json
import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

In [8]:
"""Globals/Constants"""
SAMPLE_SIZE = 200000
RANDOM_STATE = 42

In [9]:
"""Load Data"""
cleaned = pd.read_pickle('cleaned_dataset.pkl')

In [10]:
"""Sample for Graph Analysis"""
if SAMPLE_SIZE is not None and len(cleaned) > SAMPLE_SIZE:
    sample = cleaned.sample(n = SAMPLE_SIZE, random_state = RANDOM_STATE).copy()
else:
    sample = cleaned.copy()

print(f"Working sample size: {len(sample):,}")

Working sample size: 200,000


In [11]:
"""Build Citation Network"""
paper_ids = set(sample['id'])

citation_graph = nx.DiGraph()
citation_graph.add_nodes_from(sample['id'])

edge_count = 0
for paper in sample.itertuples():
    for ref_id in paper.references:
        if ref_id in paper_ids:
            citation_graph.add_edge(paper.id, ref_id)
            edge_count += 1

print(f"Nodes: {citation_graph.number_of_nodes():,}")
print(f"Edges: {citation_graph.number_of_edges():,}")
print(f"Is DAG: {nx.is_directed_acyclic_graph(citation_graph)}")

Nodes: 200,000
Edges: 106,762
Is DAG: False


In [15]:
"""Graph Metrics: In-Degree Centrality and PageRank"""
in_degree = dict(citation_graph.in_degree())
pagerank = nx.pagerank(citation_graph, alpha = 0.85)
id_to_info = sample.set_index('id')[['title', 'year', 'venue', 'n_citation']].to_dict('index')

rows = []
for paper_id, pr_score in pagerank.items():
    info = id_to_info.get(paper_id, {})
    rows.append({'id': paper_id, 'title': info.get('title', ''), 'year': info.get('year', ''), 'venue': info.get('venue', ''), 'n_citation': info.get('n_citation', 0), 'in_degree': in_degree.get(paper_id, 0), 'pagerank': pr_score})

metrics_df = pd.DataFrame(rows)
top_papers = metrics_df.sort_values('pagerank', ascending = False).head(20).reset_index(drop = True)
top_papers

,id,title,year,venue,n_citation,in_degree,pagerank
0,50dd56db-151d-4d62-8576-65f0ef6f381b,Support-Vector Networks,1995,Machine Learning,26114,459,0.001421
1,f14df1ed-e3e9-4348-9040-fc06e3411b95,"Pastry: Scalable, Decentralized Object Locatio...",2001,Lecture Notes in Computer Science,10467,265,0.000880
2,b68fc787-7817-421e-8e66-8a98ab9db1ad,Handbook of Applied Cryptography,1996,,18201,235,0.000816
3,c061069f-29d1-46d4-9974-dede8d5461f9,Genetic programming: on the programming of com...,1992,,15096,256,0.000749
4,510eec1d-f82c-4b19-b116-b8fd4c66531a,Computational geometry: an introduction,1985,Mathematics of Computation,2862,189,0.000707
5,74e3fd8b-f955-4fde-aad8-0a705f05e27e,The temporal logic of programs,1977,foundations of computer science,3545,129,0.000633
6,9849d9c4-a97f-452f-882c-42a8c6cab0b5,A Temporal Logic of Nested Calls and Returns,2004,tools and algorithms for construction and anal...,3137,229,0.000603
7,b592576f-ff29-4a68-9b2f-8a8ad02e9c70,Shape matching and object recognition using sh...,2002,IEEE Transactions on Pattern Analysis and Mach...,6365,197,0.000585
8,1545dfd3-2c25-4ff1-b43c-df4a2a501d06,GPSR: greedy perimeter stateless routing for w...,2000,acm ieee international conference on mobile co...,8821,202,0.000574
9,0ddbfee1-8cc2-49f6-be79-59276f496884,"Independent component analysis, a new concept?",1994,Signal Processing,9460,150,0.000570
